In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
import os
import sys
import logging
from argparse import Namespace

# COMMAND ----------
# Define Widgets for User Input
dbutils.widgets.text("snapshot", "202505", "Snapshot (YYYYMM)")
dbutils.widgets.text("DIL", "DIL1", "DIL Value")
dbutils.widgets.dropdown("mode", "dummy", ["dummy", "real"], "Training Mode")

# Retrieve Parameters
SNAPSHOT = dbutils.widgets.get("snapshot")
DIL = dbutils.widgets.get("DIL")
MODE = dbutils.widgets.get("mode")

# Add the repo's src directory to the Python path
REPO_ROOT = os.path.abspath("/Workspace/Users/hanif.m.abidin@gmail.com/amfs_telemarketing_new_acquisition")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from lib.train.train_prep import prepare_training_data
from lib.train.model_trainer import train_model

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define Configuration Paths
CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
MAIN_CONFIG = os.path.join(CONFIG_DIR, "main_config.yaml")
TRAIN_CONFIG = os.path.join(CONFIG_DIR, "train_config.yaml")
MODEL_CONFIG = os.path.join(CONFIG_DIR, "model_config.yaml")

# Create mock args object
args = Namespace(snapshot=SNAPSHOT, dil=DIL, mode=MODE)

# COMMAND ----------
import os
import sys
import logging
from argparse import Namespace

# Add the repo's src directory to the Python path
REPO_ROOT = os.path.abspath("/Workspace/Users/hanif.m.abidin@gmail.com/amfs_telemarketing_new_acquisition")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from lib.train.train_prep import prepare_training_data
from lib.train.model_trainer import train_model

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define Configuration Paths
CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
MAIN_CONFIG = os.path.join(CONFIG_DIR, "main_config.yaml")
TRAIN_CONFIG = os.path.join(CONFIG_DIR, "train_config.yaml")
MODEL_CONFIG = os.path.join(CONFIG_DIR, "model_config.yaml")

# Create mock args object
args = Namespace(snapshot=SNAPSHOT, dil=DIL, mode=MODE)

# COMMAND ----------
# DBTITLE 1,Step 1: Data Preparation & Target Labeling
logging.info(f"Starting Training Data Preparation in {MODE} mode...")

# This handles multi-snapshot stacking if MODE is 'real'
train_data_path = prepare_training_data(MAIN_CONFIG, TRAIN_CONFIG, args)

print(f"✅ Training data prepared and saved to: {train_data_path}")

# COMMAND ----------
# DBTITLE 1,Step 2: Model Training (XGBoost)
logging.info(f"Starting Model Training for snapshot {SNAPSHOT}...")

# Fits the model, logs ROC-AUC, and saves the pickle file
model_pkl_path = train_model(MAIN_CONFIG, TRAIN_CONFIG, MODEL_CONFIG, args)

print(f"🚀 Model training complete!")
print(f"Model File: {model_pkl_path}")